# Predict Future Sales TC1 Human Baseline

This notebook is the working artifact behind `tc1_human.json`.
It is written against the task-local official data bundle under `data/tasks/competitive-data-science-predict-future-sales/data` and is organized to match the section names referenced in `provenance/action_trace.json`.


## Load Data

The resolver checks a few likely working directories so the notebook can be run either from the repo root, the task directory, or the `human_baseline/` directory.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

TASK_DATA_REL = Path("data/tasks/competitive-data-science-predict-future-sales/data")

def resolve_data_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / TASK_DATA_REL,
        cwd / "data",
        cwd.parent / "data",
    ]
    for candidate in candidates:
        if (candidate / "sales_train.csv").exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not resolve Predict Future Sales data directory from {cwd}")

DATA_DIR = resolve_data_dir()
sales = pd.read_csv(DATA_DIR / "sales_train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
items = pd.read_csv(DATA_DIR / "items.csv")
item_categories = pd.read_csv(DATA_DIR / "item_categories.csv")
shops = pd.read_csv(DATA_DIR / "shops.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print({
    "data_dir": str(DATA_DIR),
    "sales": sales.shape,
    "test": test.shape,
    "items": items.shape,
    "item_categories": item_categories.shape,
    "shops": shops.shape,
    "sample_submission": sample_submission.shape,
})


## Parse Raw Sales Date


In [ ]:
sales = sales.copy()
sales["date"] = pd.to_datetime(sales["date"], format="%d.%m.%Y")
sales["month_start"] = sales["date"].values.astype("datetime64[M]")

sales[["date", "date_block_num", "month_start"]].head()


## Trim Gross Sales Anomalies

Keep the anomaly handling conservative and aligned with the bank: only remove the obvious extreme spikes.


In [ ]:
sales = sales[(sales["item_price"] < 100000) & (sales["item_cnt_day"] < 1001)].copy()
sales[["item_price", "item_cnt_day"]].describe().T


## Complete Shop-Item-Month Grid

This is a benchmark assumption rather than a scored bank row. The implementation below builds a pair-by-month frame over all observed training pairs plus test pairs, which is closer to the winner-style month-state representation than keeping only raw observed sales rows.


In [ ]:
pair_index = pd.concat(
    [sales[["shop_id", "item_id"]], test[["shop_id", "item_id"]]],
    ignore_index=True,
).drop_duplicates()

month_index = pd.DataFrame({"date_block_num": range(35)})
pair_index["_tmp"] = 1
month_index["_tmp"] = 1
matrix = (
    pair_index.merge(month_index, on="_tmp", how="inner")
    .drop(columns="_tmp")
    [["date_block_num", "shop_id", "item_id"]]
    .sort_values(["shop_id", "item_id", "date_block_num"])
    .reset_index(drop=True)
)

matrix.shape


## Aggregate Daily Sales To Monthly Target


In [ ]:
monthly_target = (
    sales.groupby(["date_block_num", "shop_id", "item_id"], as_index=False)["item_cnt_day"]
    .sum()
    .rename(columns={"item_cnt_day": "item_cnt_month"})
)

matrix = matrix.merge(monthly_target, on=["date_block_num", "shop_id", "item_id"], how="left")
matrix[["date_block_num", "shop_id", "item_id", "item_cnt_month"]].head()


## Join Item Lookup Metadata


In [ ]:
matrix = matrix.merge(items[["item_id", "item_category_id", "item_name"]], on="item_id", how="left")
matrix[["item_id", "item_category_id", "item_name"]].head()


## Clip Monthly Target


In [ ]:
matrix["item_cnt_month"] = matrix["item_cnt_month"].fillna(0).clip(0, 20).astype(np.float32)
matrix["item_cnt_month"].describe()


## Join Item Category Lookup


In [ ]:
cats = item_categories.copy()
matrix = matrix.merge(cats, on="item_category_id", how="left")
matrix[["item_category_id", "item_category_name"]].head()


## Join Shop Lookup


In [ ]:
shop_lookup = shops.copy()
matrix = matrix.merge(shop_lookup, on="shop_id", how="left")
matrix[["shop_id", "shop_name"]].head()


## Extract City And Category Subtypes


In [ ]:
matrix["city"] = matrix["shop_name"].str.split(" ").str[0]

split_category = matrix["item_category_name"].fillna("").str.split("-")
matrix["type"] = split_category.str[0].str.strip()
matrix["subtype"] = split_category.str[1].fillna(split_category.str[0]).str.strip()

matrix[["shop_name", "city", "item_category_name", "type", "subtype"]].head()


## Encode Derived Categoricals


In [ ]:
for source_col, encoded_col in [("city", "city_code"), ("type", "type_code"), ("subtype", "subtype_code")]:
    matrix[encoded_col], _ = pd.factorize(matrix[source_col].fillna("missing"))
    matrix[encoded_col] = matrix[encoded_col].astype(np.int16)

matrix[["city_code", "type_code", "subtype_code"]].head()


## Create Shop-Item Lags


In [ ]:
def add_lag_features(frame: pd.DataFrame, value_col: str, lags: list[int], group_cols: list[str]) -> pd.DataFrame:
    result = frame.copy()
    base = frame[group_cols + ["date_block_num", value_col]].copy()
    for lag in lags:
        shifted = base.copy()
        shifted["date_block_num"] = shifted["date_block_num"] + lag
        shifted = shifted.rename(columns={value_col: f"{value_col}_lag_{lag}"})
        result = result.merge(shifted, on=group_cols + ["date_block_num"], how="left")
    return result

matrix = matrix.sort_values(["shop_id", "item_id", "date_block_num"]).reset_index(drop=True)
matrix = add_lag_features(matrix, "item_cnt_month", [1, 2, 3, 6, 12], ["shop_id", "item_id"])

lag_cols = [f"item_cnt_month_lag_{lag}" for lag in [1, 2, 3, 6, 12]]
matrix[["date_block_num", "shop_id", "item_id"] + lag_cols].head(10)


## Fill Missing Target Lags With Zero


In [ ]:
lag_cols = [f"item_cnt_month_lag_{lag}" for lag in [1, 2, 3, 6, 12]]
matrix[lag_cols] = matrix[lag_cols].fillna(0)
matrix[lag_cols].isna().sum()


## Drop Lag Warmup Months


In [ ]:
matrix = matrix[matrix["date_block_num"] > 11].copy()
matrix[["date_block_num"] + lag_cols].head()


## Create First-Sale And Last-Sale Recency Features


In [ ]:
matrix["item_shop_first_sale"] = matrix["date_block_num"] - matrix.groupby(["item_id", "shop_id"])["date_block_num"].transform("min")
matrix["item_first_sale"] = matrix["date_block_num"] - matrix.groupby("item_id")["date_block_num"].transform("min")

matrix["nonzero_sale_month"] = matrix["date_block_num"].where(matrix["item_cnt_month"] > 0)
matrix["prev_nonzero_sale_month"] = matrix.groupby(["shop_id", "item_id"])["nonzero_sale_month"].transform(lambda s: s.ffill().shift(1))
matrix["item_shop_last_sale"] = (matrix["date_block_num"] - matrix["prev_nonzero_sale_month"]).fillna(-1).astype(np.int16)

matrix[[
    "date_block_num",
    "shop_id",
    "item_id",
    "item_shop_last_sale",
    "item_shop_first_sale",
    "item_first_sale",
]].head(10)


## Grouped Mean History Families

The notebook corpus often adds many more lagged grouped means such as month-item, month-shop, and month-category averages. Those branches are deliberately omitted from the current conservative action bank, so they are not part of `tc1_human.json`.


## Price Trend Features

Price-trend and revenue-trend branches are likewise common in top notebooks, but they are not currently selected in the human baseline because the phase-2 bank intentionally excluded them.
